In [0]:
# Scratch notebook used by the Lakeflow Connect ingestion wizard and assistant to execute operations
# supported by the ingestion_agent Spark API.
# Cells below are appended automatically, one per operation. Safe to delete when the
# wizard and assistant are not running; it will be recreated on demand.

In [0]:
import json
OPTIONS = {"connectionType":"KAFKA","operation":"list_operations"}
SETUP_ERROR_CODES = ["SOURCE_NOT_REGISTERED","SOURCE_DISABLED","CONFLICTING_OPTIONS","AMBIGUOUS_SOURCE","UNKNOWN_OPTIONS","MISSING_REQUIRED_OPTIONS","OPERATION_NOT_SUPPORTED","CONNECTION_RESOLUTION_FAILED"]

# OPTIONS is injected by buildSparkAgentScript in sparkExecutor.ts and carries the connection
# identity (connectionName for a concrete operation, or connectionType for list_operations
# discovery) alongside the operation and its params. SETUP_ERROR_CODES is injected alongside it
# from the frontend's SparkSetupErrorCode enum. Do NOT redefine either here.
#
# Prints one JSON envelope. A failure carries a bounded code the frontend records in telemetry:
# `setup_error_code` when the data source rejects the read before the operation runs, or
# `operation_error_code` when the operation reports an error through its `_meta` row. `error` is
# the raw message, which stays in this notebook's output; the frontend never records it.
import re

# The data source prefixes its setup code onto the message as "[CODE] ...", ahead of any other
# bracketed text the message carries (option lists, a wrapped cause's error class). Only that first
# bracketed token is read, and only a code in the injected SETUP_ERROR_CODES counts, so text echoed
# later in a message can't surface as a code. An unrecognized first token falls through to the
# generic failure, never to a skip.
_FIRST_BRACKETED_TOKEN = re.compile(r"\[([^\]]*)\]")

# Runtimes built before the data source tagged its messages still raise these two untagged; match
# their exact legacy wording rather than a looser phrase.
_UNTAGGED_SETUP_ERRORS = (
    ("No agent source registered for '", "SOURCE_NOT_REGISTERED"),
    ("The 'ingestion_agent' DataSource is disabled.", "SOURCE_DISABLED"),
)


def _setup_error_code(message):
    match = _FIRST_BRACKETED_TOKEN.search(message)
    if match and match.group(1) in SETUP_ERROR_CODES:
        return match.group(1)
    for text, code in _UNTAGGED_SETUP_ERRORS:
        if text in message:
            return code
    return None


try:
    reader = spark.read.format("ingestion_agent")
    for _k, _v in OPTIONS.items():
        reader = reader.option(_k, _v)
    df = reader.load()
    rows = []
    operation_error = None
    for row in df.collect():
        values = row.asDict(recursive=True)
        meta = values.get("_meta") or {}
        if meta.get("status") == "error":
            # A Metadata-kind operation reports a failure as a single error row rather than raising.
            operation_error = meta
            break
        rows.append({k: v for k, v in values.items() if k != "_meta"})
    if operation_error is None:
        print(json.dumps({"success": True, "rows": rows}, default=str))
    else:
        print(
            json.dumps(
                {
                    "success": False,
                    "operation_error_code": operation_error.get("code"),
                    "error": operation_error.get("message"),
                },
                default=str,
            )
        )
except Exception as e:
    message = str(e)
    print(
        json.dumps(
            {"success": False, "setup_error_code": _setup_error_code(message), "error": message},
            default=str,
        )
    )
